In [ ]:
import cv2
import numpy as np

# Lists of Hue, Saturation & Brightness Values ( First 3 = Dark, Last 3 = Light )

myColors = [
    [0, 100, 100, 10, 255, 255],     
    [100, 100, 100, 130, 255, 255], 
    [35, 60, 60, 85, 255, 255],     
    [11, 100, 100, 25, 255, 255],  
    [26, 80, 80, 34, 255, 255],     
    [130, 60, 60, 160, 255, 255], 
    [161, 60, 60, 179, 255, 255]    
]

# Lists of Rgb Color Values

myColorValues = [
    (0, 0, 255),     
    (255, 0, 0),      
    (0, 255, 0),     
    (0, 165, 255),  
    (0, 255, 255),  
    (255, 0, 128),   
    (180, 105, 255)  
]


myPoints = [] # The Coordinates and Color Indices of the Tips of the Markers
missing_frame_counter = 0 # The Number of Frames Recording the Marker's Absence

def findColor(img, myColors, myColorValues):
    global missing_frame_counter
    imgHSV = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) # Preventing the Computer from Confusing Colors
    newPoints = []
    marker_found_this_frame = False

    for count, color in enumerate(myColors):
        lower = np.array(color[0:3])
        upper = np.array(color[3:6])
        mask = cv2.inRange(imgHSV, lower, upper) # Creates a Mask with Dark / Light Hsv Values | If a Pixel fits = White, If a Pixel doesn't fit = Black |
        
        x, y = getContours(mask)
        if x != 0 and y != 0:
            newPoints.append([x, y, count])
            marker_found_this_frame = True

    if marker_found_this_frame:
        missing_frame_counter = 0 
    else:
        missing_frame_counter += 1 

    return newPoints

def getContours(img):
    contours, hierarchy = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    x, y, w, h = 0, 0, 0, 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 500: 
            perimeter = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02 * perimeter, True)
            x, y, w, h = cv2.boundingRect(approx)
            
    
    return x + w // 2, y

def drawOnCanvas(myPoints, imgResult):
    for point in myPoints:
        cv2.circle(imgResult, (point[0], point[1]), 10, myColorValues[point[2]], cv2.FILLED)

try:
    webcam = cv2.VideoCapture(1) 
    webcam.set(3, 640)
    webcam.set(4, 480)
    webcam.set(10, 150) 
    while True:
        success, img = webcam.read()
        if not success:
            print("Failed to Read Valid Frames from the Webcam")
            break
            
        img = cv2.flip(img, 1)
        imgResult = img.copy()
        
        newPoints = findColor(img, myColors, myColorValues)
        if len(newPoints) != 0:
            for newP in newPoints:
                myPoints.append(newP)
                
        if missing_frame_counter > 15:
            myPoints.clear()

        drawOnCanvas(myPoints, imgResult)
        cv2.imshow("Virtual Paint Canvas", imgResult)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    webcam.release()
    cv2.destroyAllWindows()

except BaseException as e:
    print(f"Failed to Access your Webcam, An Error has Occured = {e}")